# MDR-TS v9.2
**Temporal-Station Baseline Model for Soil Moisture Prediction**

**Author:** Jakob Balkovec  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR
**Notebook Type:** Training & Evaluation  
**Last Updated:** Tue Jan 6th 2026

---

## Model Summary
- **Model Name:** MDR-TS  
- **Version:** v9.1
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/base_1.0/split_meta.json`
- **Environment:** Google Colab / VS Code Remote Kernel

---

Adapted for a Macbook M2 Pro environment.

**What's new?**

- Builds directly on `v9.1.0`, which introduced a hard-gated two-expert model separating dry and wet regimes using a rain-based gate.
- Retains the same temporal split, base feature set, and gating logic to preserve comparability and avoid confounding effects.
- Introduces rain impulse features (`rain_event_impulse_0_7`, `rain_mm_impulse_0_7`, `days_since_rain_event`) exclusively to `Expert B` (wet regime).
- `Expert A` (dry regime) remains unchanged to serve as a stable reference and prevent performance regression on dry periods.
- The goal is to improve wet-period modeling capacity without diluting dry-regime performance or inflating overall model complexity.


## 0. Imports

In [1]:
import os
import random
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML / Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Gradient Boosting (baseline model)
from xgboost import XGBRegressor

# PyTorch
import torch

# SciPy
from scipy.special import expit  # sigmoid

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"using: {device}")

imports loaded
using: cpu


## 1. Environment Setup

In [2]:
# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Random seed set to {SEED}")

# Environment / Runtime Info
def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    # Colab-specific checks
    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print("  GPU available: False")

print_env_info()

# Plotting defaults
plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

Random seed set to 42
Environment information:
  Python version: 3.10.18
  NumPy version:  1.26.4
  Pandas version: 2.0.3
  XGBoost version: 2.1.2
  Running in Colab: False
  GPU available: False
environment setup complete


## 2. Data Access

In [3]:
# Project paths
VERSION = "v9"
SUBVERSION = "v9.1"
RUN_NAME = "mdr_ts_v9_1"

PROJECT_ROOT = "/Users/jbalkovec/Desktop/MDR"
DATA_ROOT = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT = f"{DATA_ROOT}/splits"
OUTPUT_ROOT = f"{PROJECT_ROOT}/Models/Temporal/{VERSION}/{SUBVERSION}"

# Create output directory if missing
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",
      os.path.exists(DATA_ROOT))
print("  splits exists:",
      os.path.exists(SPLIT_ROOT))
print("  output exists:",
      os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: /Users/jbalkovec/Desktop/MDR
  DATA_ROOT:    /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data
  SPLIT_ROOT:   /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  /Users/jbalkovec/Desktop/MDR/Models/Temporal/v9/v9.1

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


## 3. Data Loading

In [4]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_3.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_3.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_3.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

DROP_COLS = ["slope", "elev"]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    cols_present = [c for c in DROP_COLS if c in d.columns]
    d.drop(columns=cols_present, inplace=True)
    print(f"{name}: dropped columns {cols_present}")
    print(f"\n{name}: shape={d.shape}")
    print(f"{name}: columns={len(d.columns)}")

Split files:
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/train_derived_new.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/val_derived_new.csv
  /Users/jbalkovec/Desktop/MDR/Temporal/Pipeline/data/splits/derived_new/test_derived_new.csv
train: dropped columns ['slope', 'elev']

train: shape=(16972, 352)
train: columns=352
val: dropped columns ['slope', 'elev']

val: shape=(2919, 352)
val: columns=352
test: dropped columns ['slope', 'elev']

test: shape=(2829, 352)
test: columns=352


In [5]:
# Dropping TouchNet because it has no rows in the validation split

DROP_STATION = "Touchet_WA_824"

def drop_station(df, station_id=DROP_STATION):
    before = len(df)
    out = df[df["station_id"] != station_id].copy()
    after = len(out)
    print(f"Dropped {station_id}: {before} -> {after} rows (-{before-after})")
    return out

train_df = drop_station(train_df)
val_df   = drop_station(val_df)
test_df  = drop_station(test_df)

Dropped Touchet_WA_824: 16972 -> 13661 rows (-3311)
Dropped Touchet_WA_824: 2919 -> 2919 rows (-0)
Dropped Touchet_WA_824: 2829 -> 2659 rows (-170)


In [7]:
print("Common columns across splits:",
      len(set(train_df.columns) & set(val_df.columns) & set(test_df.columns)))

print("\ncolumns:")
print(list(train_df.columns)[:30])

Common columns across splits: 352

columns:
['station_id', 'date', 'longitude', 'latitude', 'precip_mm', 's1_vv', 's1_vh', 's2_b4', 's2_b8', 's2_b11', 's2_b12', 'LST_modis', 'aspect', 'DOY', 'soil_moisture_5cm', 'F_NDVI', 'F_NDMI', 'F_MSI', 'E_SAR_ratio', 'E_SAR_diff', 'G_API', 'G_DSLR', 'G_rain_sum_3d', 'G_rain_sum_7d', 'G_rain_sum_30d', 'A_d_G_API_kobs1', 'A_d_G_API_kobs2', 'A_d_G_API_kobs5', 'A_d_G_API_kobs7', 'A_d_G_API_kobs14']


## 4. Data Sanity Checks

In [ ]:
# Column definitions (baseline)
TARGET_COL = "soil_moisture_5cm"
KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]
FEATURE_COLS = [
  "precip_mm",

  "G_rain_sum_3d",
  "G_rain_sum_7d",
  "G_rain_sum_30d",
  "G_API",

  "G_DSLR",

  "C_lag_E_SAR_diff_kobs12",
  "C_lag_E_SAR_diff_kobs30",
  "C_lag_E_SAR_ratio_kobs30",
  "C_lag_F_NDVI_kobs30",
  "C_lag_LST_modis_kobs12",
  "C_lag_LST_modis_kobs30",
  "DOY",
  "D_sa_E_SAR_ratio",
  "D_sa_F_NDMI",
  "D_z_E_SAR_ratio",
  "D_z_F_NDMI",
  "E_SAR_diff",
  "E_SAR_ratio",
  "F_MSI",
  "F_NDMI",
  "V_ema_LST_modis_kobs30",
  "V_rollmax_E_SAR_diff_kobs14",
  "V_rollmax_E_SAR_diff_kobs30",
  "V_rollmax_F_NDVI_kobs30",
  "V_rollmax_G_API_kobs30",
  "V_rollmax_G_API_kobs7",
  "V_rollmax_LST_modis_kobs7",
  "V_rollmax_s2_b11_kobs30",
  "V_rollmean_G_API_kobs30",
  "V_rollmin_E_SAR_diff_kobs30",
  "V_rollmin_E_SAR_ratio_kobs30",
  "V_rollmin_F_NDMI_kobs30",
  "V_rollmin_G_API_kobs7",
  "V_rollmin_s2_b11_kobs30",
  "V_rollmin_s2_b12_kobs30",
  "s1_vh",
  "s2_b8",
  "A_d_LST_modis_kobs7",
  "A_grad_LST_modis_kobs14",
  "C_lag_E_SAR_diff_kobs6",
  "V_rollmax_E_SAR_diff_kobs7",
  "V_rollmax_E_SAR_ratio_kobs14",
  "V_rollmax_F_NDVI_kobs14",
  "s2_b12",
  "D_sa_LST_modis"
]

# quick validation
expected = set(KEEP_META_COLS + FEATURE_COLS + [TARGET_COL])
missing_train = sorted(list(expected - set(train_df.columns)))
missing_val   = sorted(list(expected - set(val_df.columns)))
missing_test  = sorted(list(expected - set(test_df.columns)))

if missing_train or missing_val or missing_test:
    raise ValueError(
        f"Missing columns:\n"
        f"  train: {missing_train}\n"
        f"  val:   {missing_val}\n"
        f"  test:  {missing_test}"
    )

print("Columns locked")
print("  Features:", len(FEATURE_COLS))
print("  Target:  ", TARGET_COL)

Columns locked
  Features: 46
  Target:   soil_moisture_5cm


In [9]:
for d in (train_df, val_df, test_df):
    d["date"] = pd.to_datetime(d["date"], errors="coerce")

stations = sorted(set(train_df["station_id"].dropna().unique())
                  | set(val_df["station_id"].dropna().unique())
                  | set(test_df["station_id"].dropna().unique()))

print("\n=== TEMPORAL LEAKAGE CHECKS (per station) ===")
bad = 0

for sid in stations:
    tr = train_df[train_df["station_id"] == sid]["date"].dropna()
    va = val_df[val_df["station_id"] == sid]["date"].dropna()
    te = test_df[test_df["station_id"] == sid]["date"].dropna()

    if len(tr) == 0 or len(va) == 0 or len(te) == 0:
        print(f"[WARN] station {sid}: missing split data (train={len(tr)}, val={len(va)}, test={len(te)})")
        bad += 1
        continue

    tr_min, tr_max = tr.min(), tr.max()
    va_min, va_max = va.min(), va.max()
    te_min, te_max = te.min(), te.max()

    # date overlap checks (hard leakage)
    overlap_tr_va = len(set(tr.unique()) & set(va.unique()))
    overlap_tr_te = len(set(tr.unique()) & set(te.unique()))
    overlap_va_te = len(set(va.unique()) & set(te.unique()))

    # ordering check (soft but important)
    order_ok = (tr_max < va_min) and (va_max < te_min)

    if overlap_tr_va or overlap_tr_te or overlap_va_te or (not order_ok):
        print(f"[ERROR] station {sid}:")
        print(f"  train: {tr_min} -> {tr_max}")
        print(f"  val:   {va_min} -> {va_max}")
        print(f"  test:  {te_min} -> {te_max}")
        print(f"  overlaps: train∩val={overlap_tr_va}, train∩test={overlap_tr_te}, val∩test={overlap_va_te}")
        print(f"  order_ok: {order_ok}")
        bad += 1
    else:
        print(f"[OK] station {sid}: train<{val_df is not None and 'val' or ''}val<test with no date overlap")

if bad == 0:
    print("\n[INFO] No temporal leakage detected.")
else:
    print(f"\n[WARNING] {bad} station(s) have temporal leakage or split issues.")



=== TEMPORAL LEAKAGE CHECKS (per station) ===
[OK] station Darrington: train<valval<test with no date overlap
[OK] station Quinault: train<valval<test with no date overlap
[OK] station SourdoughGulch_WA_985: train<valval<test with no date overlap
[OK] station Spokane: train<valval<test with no date overlap

[INFO] No temporal leakage detected.


In [10]:
# yoinked from v8.2

RAIN_COL = "precip_mm"
RAIN_THR = 4.0
K = 7
WEIGHTS = np.array([1.0, 0.6, 0.2, 0.1, 0.05, 0.02, 0.01, 0.0], dtype=float)  # len = K+1

NEW_FEATURES = [
    "rain_event_impulse_0_7",
    "rain_mm_impulse_0_7",
    "days_since_rain_event",
]

def add_rain_impulse_features(d: pd.DataFrame) -> pd.DataFrame:
    d = d.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d.sort_values(["station_id", "date"]).reset_index(drop=True)

    d["_rain_event"] = (d[RAIN_COL] >= RAIN_THR).astype(int)

    for k in range(0, K + 1):
        d[f"_ev_lag{k}"] = d.groupby("station_id")["_rain_event"].shift(k)
        d[f"_mm_lag{k}"] = d.groupby("station_id")[RAIN_COL].shift(k)

    ev_cols = [f"_ev_lag{k}" for k in range(0, K + 1)]
    mm_cols = [f"_mm_lag{k}" for k in range(0, K + 1)]

    ev_mat = d[ev_cols].fillna(0).to_numpy(dtype=float)
    mm_mat = d[mm_cols].fillna(0).to_numpy(dtype=float)

    d["rain_event_impulse_0_7"] = (ev_mat * WEIGHTS).sum(axis=1)
    d["rain_mm_impulse_0_7"]    = (mm_mat * WEIGHTS).sum(axis=1)

    def _days_since_event(group: pd.DataFrame) -> pd.Series:
        ev = group["_rain_event"].to_numpy()
        out = np.full(len(ev), np.nan, dtype=float)
        last_idx = None
        for i in range(len(ev)):
            if ev[i] == 1:
                last_idx = i
                out[i] = 0.0
            else:
                if last_idx is not None:
                    out[i] = float(i - last_idx)
        return pd.Series(out, index=group.index)

    d["days_since_rain_event"] = (
        d.groupby("station_id", group_keys=False)
         .apply(_days_since_event)
         .clip(upper=30)
    )

    drop_cols = ["_rain_event"] + ev_cols + mm_cols
    d.drop(columns=drop_cols, inplace=True, errors="ignore")
    return d

train_df = train_df.copy(); val_df = val_df.copy(); test_df = test_df.copy()
train_df["_split"] = "train"
val_df["_split"]   = "val"
test_df["_split"]  = "test"

all_df = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)
all_df = add_rain_impulse_features(all_df)

train_df = all_df[all_df["_split"] == "train"].drop(columns=["_split"]).reset_index(drop=True)
val_df   = all_df[all_df["_split"] == "val"].drop(columns=["_split"]).reset_index(drop=True)
test_df  = all_df[all_df["_split"] == "test"].drop(columns=["_split"]).reset_index(drop=True)

FEATURE_COLS_A = list(FEATURE_COLS)
FEATURE_COLS_B = list(FEATURE_COLS) + NEW_FEATURES

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    miss = [c for c in NEW_FEATURES if c not in d.columns]
    if miss:
        raise ValueError(f"{name} missing NEW_FEATURES: {miss}")

print("v9.2 impulse features added (wet-expert-only)")
print("  baseline features (A):", len(FEATURE_COLS_A))
print("  wet expert features (B):", len(FEATURE_COLS_B))
print("  NaN rate (train/val/test):")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(" ", name, d[NEW_FEATURES].isna().mean().round(4).to_dict())

v9.2 impulse features added (wet-expert-only)
  baseline features (A): 46
  wet expert features (B): 49
  NaN rate (train/val/test):
  train {'rain_event_impulse_0_7': 0.0, 'rain_mm_impulse_0_7': 0.0, 'days_since_rain_event': 0.0027}
  val {'rain_event_impulse_0_7': 0.0, 'rain_mm_impulse_0_7': 0.0, 'days_since_rain_event': 0.0}
  test {'rain_event_impulse_0_7': 0.0, 'rain_mm_impulse_0_7': 0.0, 'days_since_rain_event': 0.0}


In [11]:
print("\nTemporal split check (train -> validation):")

for sid in sorted(train_df["station_id"].unique()):
    train_dates = train_df.loc[train_df["station_id"] == sid, "date"]
    val_dates   = val_df.loc[val_df["station_id"] == sid, "date"]

    max_train = train_dates.max()
    min_val   = val_dates.min()

    print(f"  Station {sid}:")
    print(f"    train max date: {max_train}")
    print(f"    val   min date: {min_val}")


Temporal split check (train -> validation):
  Station Darrington:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station Quinault:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station SourdoughGulch_WA_985:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00
  Station Spokane:
    train max date: 2021-09-23 00:00:00
    val   min date: 2021-09-24 00:00:00


## 5. Train / Validation / Test Split

### Split Strategy
- **Training set:**  
  Two stations, early time period  
- **Validation set:**  
  Same stations as training, held-out **future dates** (temporal holdout)
- **Test set:**  
  One completely unseen station (station-level holdout)

### Motivation
- Validation evaluates **temporal generalization** on known stations
- Test evaluates **spatial generalization** to an unseen station
- This avoids spatial leakage while preserving sufficient training data

In [12]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train ∩ test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   ∩ test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")


=== SPLIT SUMMARY ===

TRAIN
  rows:     13661
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2011-10-06 00:00:00 -- 2021-09-23 00:00:00

VAL
  rows:     2919
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2021-09-24 00:00:00 -- 2023-11-12 00:00:00

TEST
  rows:     2659
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2023-11-13 00:00:00 -- 2025-12-31 00:00:00

=== LEAKAGE CHECK ===
train ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
val   ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']

-- split locked --


## 6. Model Definition

### 6.1 Feature Matrix Construction

In [13]:
y_train = train_df[TARGET_COL].to_numpy()
y_val   = val_df[TARGET_COL].to_numpy()
y_test  = test_df[TARGET_COL].to_numpy()

meta_train = train_df[KEEP_META_COLS].copy() # dbg
meta_val   = val_df[KEEP_META_COLS].copy()   # dbg
meta_test  = test_df[KEEP_META_COLS].copy()  # dbg

X_train_A = train_df[FEATURE_COLS_A].copy()
X_val_A   = val_df[FEATURE_COLS_A].copy()
X_test_A  = test_df[FEATURE_COLS_A].copy()

X_train_B = train_df[FEATURE_COLS_B].copy()
X_val_B   = val_df[FEATURE_COLS_B].copy()
X_test_B  = test_df[FEATURE_COLS_B].copy()

# sanity
assert X_train_A.shape[0] == len(y_train) == len(meta_train)
assert X_val_A.shape[0]   == len(y_val)   == len(meta_val)
assert X_test_A.shape[0]  == len(y_test)  == len(meta_test)

assert X_train_B.shape[0] == len(y_train)
assert X_val_B.shape[0]   == len(y_val)
assert X_test_B.shape[0]  == len(y_test)

print("v9.2 matrices ready")
print("  A features:", X_train_A.shape[1], "B features:", X_train_B.shape[1])
print("  X_train_A:", X_train_A.shape, "X_train_B:", X_train_B.shape)
print("  X_val_A:  ", X_val_A.shape,   "X_val_B:  ", X_val_B.shape)
print("  X_test_A: ", X_test_A.shape,  "X_test_B: ", X_test_B.shape)

v9.2 matrices ready
  A features: 46 B features: 49
  X_train_A: (13661, 46) X_train_B: (13661, 49)
  X_val_A:   (2919, 46) X_val_B:   (2919, 49)
  X_test_A:  (2659, 46) X_test_B:  (2659, 49)


In [118]:
# starting simple: existing feature/s as the gating signal
GATE_COL = "G_rain_sum_7d"   # maybe: "G_API"
GATE_MODE = "quantile"       # "quantile" or "fixed"
WET_Q = 0.55                 # top 25% = wet (only used if GATE_MODE="quantile")
WET_THR = 4.0                # only used if GATE_MODE="fixed"

if GATE_MODE == "quantile":
    thr = float(train_df[GATE_COL].quantile(WET_Q))
elif GATE_MODE == "fixed":
    thr = float(WET_THR)
else:
    raise ValueError(f"Unknown GATE_MODE: {GATE_MODE}")

train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

for d in (train_df, val_df, test_df):
    d["is_wet"] = (d[GATE_COL] >= thr).astype(int)

print("Gate ready")
print(f"  GATE_COL: {GATE_COL}")
print(f"  mode:     {GATE_MODE}")
print(f"  thr:      {thr:.4f}")
print("  wet% train/val/test:",
      train_df["is_wet"].mean().round(3),
      val_df["is_wet"].mean().round(3),
      test_df["is_wet"].mean().round(3))

Gate ready
  GATE_COL: G_rain_sum_7d
  mode:     quantile
  thr:      21.0000
  wet% train/val/test: 0.451 0.45 0.453


In [105]:
def _metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

### 6.4 Expert A (Model A) | DRY

In [119]:
A_train_df = train_df[train_df["is_wet"] == 0].copy()
A_val_df   = val_df[val_df["is_wet"] == 0].copy()

XA_train = A_train_df[FEATURE_COLS_A].copy()
yA_train = A_train_df[TARGET_COL].to_numpy()

XA_val   = A_val_df[FEATURE_COLS_A].copy()
yA_val   = A_val_df[TARGET_COL].to_numpy()

metaA_train = A_train_df[KEEP_META_COLS].copy()
metaA_val   = A_val_df[KEEP_META_COLS].copy()

print("Expert A (DRY) matrices ready [v9.2]")
print("  train rows:", len(A_train_df), "wet%:", A_train_df["is_wet"].mean())
print("  val rows:  ", len(A_val_df),   "wet%:", A_val_df["is_wet"].mean())
print("  XA_train:", XA_train.shape, "yA_train:", yA_train.shape)
print("  XA_val:  ", XA_val.shape,   "yA_val:  ", yA_val.shape)

Expert A (DRY) matrices ready [v9.2]
  train rows: 7502 wet%: 0.0
  val rows:   1604 wet%: 0.0
  XA_train: (7502, 46) yA_train: (7502,)
  XA_val:   (1604, 46) yA_val:   (1604,)


In [120]:
xgbA = XGBRegressor(
    subsample=0.9,
    reg_lambda=2.0,
    reg_alpha=0.05,
    n_estimators=4000,
    min_child_weight=3,
    max_depth=7,
    learning_rate=0.05,
    gamma=0.0,
    colsample_bytree=0.75,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42,
)

rfA = RandomForestRegressor(
    n_estimators=800,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.5,
    max_depth=16,
    n_jobs=-1,
    random_state=42,
)

xgbA.fit(XA_train, yA_train)
rfA.fit(XA_train, yA_train)

# meta learner (ridge) trained on base preds
A_pred_train = np.vstack([xgbA.predict(XA_train), rfA.predict(XA_train)]).T
A_pred_val   = np.vstack([xgbA.predict(XA_val),   rfA.predict(XA_val)]).T

ridgeA = Ridge(alpha=1.0, random_state=42)
ridgeA.fit(A_pred_train, yA_train)

# evaluate on DRY val
yA_hat_val = ridgeA.predict(A_pred_val)

r2_A  = r2_score(yA_val, yA_hat_val)
mae_A = mean_absolute_error(yA_val, yA_hat_val)
rmse_A = np.sqrt(mean_squared_error(yA_val, yA_hat_val))

print("Expert A (DRY) trained")
print("  val metrics (DRY-only):")
print(f"    R2  : {r2_A:.6f}")
print(f"    MAE : {mae_A:.6f}")
print(f"    RMSE: {rmse_A:.6f}")
print("  ridge weights:", ridgeA.coef_.round(6), "intercept:", float(ridgeA.intercept_))

Expert A (DRY) trained
  val metrics (DRY-only):
    R2  : 0.818629
    MAE : 0.032849
    RMSE: 0.043020
  ridge weights: [0.650981 0.352414] intercept: -0.0005370557526763742


In [121]:
A_base_val  = np.vstack([xgbA.predict(X_val_A),  rfA.predict(X_val_A)]).T
A_base_test = np.vstack([xgbA.predict(X_test_A), rfA.predict(X_test_A)]).T

yhatA_val  = ridgeA.predict(A_base_val)
yhatA_test = ridgeA.predict(A_base_test)

print("Expert A full-split predictions ready [v9.2]")
print("  yhatA_val :", yhatA_val.shape,  "min/mean/max:", float(np.min(yhatA_val)), float(np.mean(yhatA_val)), float(np.max(yhatA_val)))
print("  yhatA_test:", yhatA_test.shape, "min/mean/max:", float(np.min(yhatA_test)), float(np.mean(yhatA_test)), float(np.max(yhatA_test)))

Expert A full-split predictions ready [v9.2]
  yhatA_val : (2919,) min/mean/max: 0.01677222819579028 0.18791046485256854 0.3390233030859695
  yhatA_test: (2659,) min/mean/max: 0.016379572293142675 0.18935711363420463 0.3329754225942216


In [122]:
r2v, maev, rmsev = _metrics(y_val, yhatA_val)
r2t, maet, rmset = _metrics(y_test, yhatA_test)

print("Expert A (DRY) sanity on FULL splits (not gated):")
print(f"  VAL : R2={r2v:.6f} MAE={maev:.6f} RMSE={rmsev:.6f}")
print(f"  TEST: R2={r2t:.6f} MAE={maet:.6f} RMSE={rmset:.6f}")

val_is_dry  = (val_df["is_wet"].to_numpy() == 0)
test_is_dry = (test_df["is_wet"].to_numpy() == 0)

r2v_d, maev_d, rmsev_d = _metrics(y_val[val_is_dry], yhatA_val[val_is_dry])
r2t_d, maet_d, rmset_d = _metrics(y_test[test_is_dry], yhatA_test[test_is_dry])

print("Expert A (DRY) on DRY-only slices:")
print(f"  VAL  dry : R2={r2v_d:.6f} MAE={maev_d:.6f} RMSE={rmsev_d:.6f}  (n={val_is_dry.sum()})")
print(f"  TEST dry : R2={r2t_d:.6f} MAE={maet_d:.6f} RMSE={rmset_d:.6f}  (n={test_is_dry.sum()})")

Expert A (DRY) sanity on FULL splits (not gated):
  VAL : R2=0.702896 MAE=0.042298 RMSE=0.054899
  TEST: R2=0.614892 MAE=0.047030 RMSE=0.059615
Expert A (DRY) on DRY-only slices:
  VAL  dry : R2=0.818629 MAE=0.032849 RMSE=0.043020  (n=1604)
  TEST dry : R2=0.769670 MAE=0.035354 RMSE=0.048243  (n=1455)


### 6.5 Expert B (Model B) | WET

In [123]:
B_train_df = train_df[train_df["is_wet"] == 1].copy()
B_val_df   = val_df[val_df["is_wet"] == 1].copy()

XB_train = B_train_df[FEATURE_COLS_B].copy()
yB_train = B_train_df[TARGET_COL].to_numpy()

XB_val   = B_val_df[FEATURE_COLS_B].copy()
yB_val   = B_val_df[TARGET_COL].to_numpy()

metaB_train = B_train_df[KEEP_META_COLS].copy()
metaB_val   = B_val_df[KEEP_META_COLS].copy()

print("Expert B (WET) matrices ready [v9.2]")
print("  train rows:", len(B_train_df), "wet%:", B_train_df["is_wet"].mean())
print("  val rows:  ", len(B_val_df),   "wet%:", B_val_df["is_wet"].mean())
print("  XB_train:", XB_train.shape, "yB_train:", yB_train.shape)
print("  XB_val:  ", XB_val.shape,   "yB_val:  ", yB_val.shape)

Expert B (WET) matrices ready [v9.2]
  train rows: 6159 wet%: 1.0
  val rows:   1315 wet%: 1.0
  XB_train: (6159, 49) yB_train: (6159,)
  XB_val:   (1315, 49) yB_val:   (1315,)


In [124]:
xgbB = XGBRegressor(
    subsample=0.9,
    reg_lambda=2.0,
    reg_alpha=0.05,
    n_estimators=4000,
    min_child_weight=3,
    max_depth=7,
    learning_rate=0.05,
    gamma=0.0,
    colsample_bytree=0.75,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42,
)

rfB = RandomForestRegressor(
    n_estimators=800,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=0.5,
    max_depth=16,
    n_jobs=-1,
    random_state=42,
)

xgbB.fit(XB_train, yB_train)
rfB.fit(XB_train, yB_train)

# meta learner (ridge) trained on base preds
B_pred_train = np.vstack([xgbB.predict(XB_train), rfB.predict(XB_train)]).T
B_pred_val   = np.vstack([xgbB.predict(XB_val),   rfB.predict(XB_val)]).T

ridgeB = Ridge(alpha=1.0, random_state=42)
ridgeB.fit(B_pred_train, yB_train)

# evaluate on WET val
yB_hat_val = ridgeB.predict(B_pred_val)

r2_B  = r2_score(yB_val, yB_hat_val)
mae_B = mean_absolute_error(yB_val, yB_hat_val)
rmse_B = np.sqrt(mean_squared_error(yB_val, yB_hat_val))

print("Expert B (WET) trained")
print("  val metrics (WET-only):")
print(f"    R2  : {r2_B:.6f}")
print(f"    MAE : {mae_B:.6f}")
print(f"    RMSE: {rmse_B:.6f}")
print("  ridge weights:", ridgeB.coef_.round(6), "intercept:", float(ridgeB.intercept_))

Expert B (WET) trained
  val metrics (WET-only):
    R2  : 0.368471
    MAE : 0.034377
    RMSE: 0.044688
  ridge weights: [0.688284 0.32211 ] intercept: -0.002665985407757532


In [125]:
B_base_val  = np.vstack([xgbB.predict(X_val_B),  rfB.predict(X_val_B)]).T
B_base_test = np.vstack([xgbB.predict(X_test_B), rfB.predict(X_test_B)]).T

yhatB_val  = ridgeB.predict(B_base_val)
yhatB_test = ridgeB.predict(B_base_test)

print("Expert B full-split predictions ready [v9.2]")
print("  yhatB_val :", yhatB_val.shape,  "min/mean/max:", float(np.min(yhatB_val)),  float(np.mean(yhatB_val)),  float(np.max(yhatB_val)))
print("  yhatB_test:", yhatB_test.shape, "min/mean/max:", float(np.min(yhatB_test)), float(np.mean(yhatB_test)), float(np.max(yhatB_test)))

Expert B full-split predictions ready [v9.2]
  yhatB_val : (2919,) min/mean/max: 0.06258816444211754 0.21532738655691858 0.3542466287186292
  yhatB_test: (2659,) min/mean/max: 0.045515557750985376 0.21617215367008458 0.33340030996525744


In [126]:
r2v_B_full, mae_v_B_full, rmse_v_B_full = _metrics(y_val, yhatB_val)
r2t_B_full, mae_t_B_full, rmse_t_B_full = _metrics(y_test, yhatB_test)

print("Expert B (WET) sanity on FULL splits (not gated):")
print(f"  VAL : R2={r2v_B_full:.6f} MAE={mae_v_B_full:.6f} RMSE={rmse_v_B_full:.6f}")
print(f"  TEST: R2={r2t_B_full:.6f} MAE={mae_t_B_full:.6f} RMSE={rmse_t_B_full:.6f}")

val_is_wet  = (val_df["is_wet"].to_numpy() == 1)
test_is_wet = (test_df["is_wet"].to_numpy() == 1)

r2v_B_wet, mae_v_B_wet, rmse_v_B_wet = _metrics(
    y_val[val_is_wet], yhatB_val[val_is_wet]
)
r2t_B_wet, mae_t_B_wet, rmse_t_B_wet = _metrics(
    y_test[test_is_wet], yhatB_test[test_is_wet]
)

print("Expert B (WET) on WET-only slices:")
print(f"  VAL  wet : R2={r2v_B_wet:.6f} MAE={mae_v_B_wet:.6f} RMSE={rmse_v_B_wet:.6f}  (n={val_is_wet.sum()})")
print(f"  TEST wet : R2={r2t_B_wet:.6f} MAE={mae_t_B_wet:.6f} RMSE={rmse_t_B_wet:.6f}  (n={test_is_wet.sum()})")

Expert B (WET) sanity on FULL splits (not gated):
  VAL : R2=0.811442 MAE=0.034365 RMSE=0.043735
  TEST: R2=0.669013 MAE=0.043379 RMSE=0.055267
Expert B (WET) on WET-only slices:
  VAL  wet : R2=0.368471 MAE=0.034377 RMSE=0.044688  (n=1315)
  TEST wet : R2=0.163596 MAE=0.042397 RMSE=0.053111  (n=1204)


## 7. Gating

In [127]:
gate_val  = val_df["is_wet"].to_numpy()
gate_test = test_df["is_wet"].to_numpy()

yhat_mix_val = gate_val * yhatB_val + (1 - gate_val) * yhatA_val
yhat_mix_test = gate_test * yhatB_test + (1 - gate_test) * yhatA_test

print("Mixture predictions ready")
print("  VAL  mix :", yhat_mix_val.shape)
print("  TEST mix:", yhat_mix_test.shape)
print("  % wet routed (val/test):",
      gate_val.mean().round(3),
      gate_test.mean().round(3))

Mixture predictions ready
  VAL  mix : (2919,)
  TEST mix: (2659,)
  % wet routed (val/test): 0.45 0.453


In [133]:
def _metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

def _print_row(name, y_true, y_pred):
    r2, mae, rmse = _metrics(y_true, y_pred)
    print(f"{name:<18} R2={r2: .6f}  MAE={mae: .6f}  RMSE={rmse: .6f}")

val_is_wet  = (val_df["is_wet"].to_numpy() == 1)
test_is_wet = (test_df["is_wet"].to_numpy() == 1)

print("=== OVERALL (FULL SPLITS) ===")
_print_row("Expert A only (VAL)", y_val, yhatA_val)
_print_row("Expert B only (VAL)", y_val, yhatB_val)
_print_row("MIX (VAL)",          y_val, yhat_mix_val)
print()
_print_row("Expert A only (TEST)", y_test, yhatA_test)
_print_row("Expert B only (TEST)", y_test, yhatB_test)
_print_row("MIX (TEST)",          y_test, yhat_mix_test)

print("\n=== WET-ONLY SLICES ===")
_print_row("Expert A on wet (VAL)", y_val[val_is_wet],  yhatA_val[val_is_wet])
_print_row("Expert B on wet (VAL)", y_val[val_is_wet],  yhatB_val[val_is_wet])
_print_row("MIX on wet (VAL)",      y_val[val_is_wet],  yhat_mix_val[val_is_wet])
print()
_print_row("Expert A on wet (TEST)", y_test[test_is_wet], yhatA_test[test_is_wet])
_print_row("Expert B on wet (TEST)", y_test[test_is_wet], yhatB_test[test_is_wet])
_print_row("MIX on wet (TEST)",      y_test[test_is_wet], yhat_mix_test[test_is_wet])

print("\n=== DRY-ONLY SLICES ===")
_print_row("Expert A on dry (VAL)", y_val[~val_is_wet],  yhatA_val[~val_is_wet])
_print_row("Expert B on dry (VAL)", y_val[~val_is_wet],  yhatB_val[~val_is_wet])
_print_row("MIX on dry (VAL)",      y_val[~val_is_wet],  yhat_mix_val[~val_is_wet])
print()
_print_row("Expert A on dry (TEST)", y_test[~test_is_wet], yhatA_test[~test_is_wet])
_print_row("Expert B on dry (TEST)", y_test[~test_is_wet], yhatB_test[~test_is_wet])
_print_row("MIX on dry (TEST)",      y_test[~test_is_wet], yhat_mix_test[~test_is_wet])

print("\nCounts (val/test):")
print("  wet:",  int(val_is_wet.sum()),  int(test_is_wet.sum()))
print("  dry:",  int((~val_is_wet).sum()), int((~test_is_wet).sum()))

=== OVERALL (FULL SPLITS) ===
Expert A only (VAL) R2= 0.702896  MAE= 0.042298  RMSE= 0.054899
Expert B only (VAL) R2= 0.811442  MAE= 0.034365  RMSE= 0.043735
MIX (VAL)          R2= 0.811061  MAE= 0.033538  RMSE= 0.043779

Expert A only (TEST) R2= 0.614892  MAE= 0.047030  RMSE= 0.059615
Expert B only (TEST) R2= 0.669013  MAE= 0.043379  RMSE= 0.055267
MIX (TEST)         R2= 0.723596  MAE= 0.038543  RMSE= 0.050505

=== WET-ONLY SLICES ===
Expert A on wet (VAL) R2=-0.401769  MAE= 0.053823  RMSE= 0.066578
Expert B on wet (VAL) R2= 0.368471  MAE= 0.034377  RMSE= 0.044688
MIX on wet (VAL)   R2= 0.368471  MAE= 0.034377  RMSE= 0.044688

Expert A on wet (TEST) R2=-0.493327  MAE= 0.061140  RMSE= 0.070966
Expert B on wet (TEST) R2= 0.163596  MAE= 0.042397  RMSE= 0.053111
MIX on wet (TEST)  R2= 0.163596  MAE= 0.042397  RMSE= 0.053111

=== DRY-ONLY SLICES ===
Expert A on dry (VAL) R2= 0.818629  MAE= 0.032849  RMSE= 0.043020
Expert B on dry (VAL) R2= 0.819317  MAE= 0.034355  RMSE= 0.042938
MIX on dry

In [129]:
g_val  = val_df[GATE_COL].to_numpy()
g_test = test_df[GATE_COL].to_numpy()

GATE_K = 1.0

wB_val  = expit(GATE_K * (g_val  - thr))
wB_test = expit(GATE_K * (g_test - thr))

yhat_soft_val  = wB_val  * yhatB_val  + (1 - wB_val)  * yhatA_val
yhat_soft_test = wB_test * yhatB_test + (1 - wB_test) * yhatA_test

print("Soft gate ready")
print("  wB val  mean/min/max:",
      wB_val.mean().round(3),
      wB_val.min().round(3),
      wB_val.max().round(3))
print("  wB test mean/min/max:",
      wB_test.mean().round(3),
      wB_test.min().round(3),
      wB_test.max().round(3))

Soft gate ready
  wB val  mean/min/max: 0.451 0.0 1.0
  wB test mean/min/max: 0.452 0.0 1.0


In [130]:
def _metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

def _print(name, y_true, y_pred):
    r2, mae, rmse = _metrics(y_true, y_pred)
    print(f"{name:<22} R2={r2: .6f}  MAE={mae: .6f}  RMSE={rmse: .6f}")

val_is_wet  = (val_df["is_wet"].to_numpy() == 1)
test_is_wet = (test_df["is_wet"].to_numpy() == 1)

print("=== OVERALL ===")
_print("Hard MIX (VAL)", y_val, yhat_mix_val)
_print("Soft MIX (VAL)", y_val, yhat_soft_val)
print()
_print("Hard MIX (TEST)", y_test, yhat_mix_test)
_print("Soft MIX (TEST)", y_test, yhat_soft_test)

print("\n=== WET ONLY ===")
_print("Hard MIX wet (VAL)", y_val[val_is_wet], yhat_mix_val[val_is_wet])
_print("Soft MIX wet (VAL)", y_val[val_is_wet], yhat_soft_val[val_is_wet])
print()
_print("Hard MIX wet (TEST)", y_test[test_is_wet], yhat_mix_test[test_is_wet])
_print("Soft MIX wet (TEST)", y_test[test_is_wet], yhat_soft_test[test_is_wet])

print("\n=== DRY ONLY ===")
_print("Hard MIX dry (VAL)", y_val[~val_is_wet], yhat_mix_val[~val_is_wet])
_print("Soft MIX dry (VAL)", y_val[~val_is_wet], yhat_soft_val[~val_is_wet])
print()
_print("Hard MIX dry (TEST)", y_test[~test_is_wet], yhat_mix_test[~test_is_wet])
_print("Soft MIX dry (TEST)", y_test[~test_is_wet], yhat_soft_test[~test_is_wet])

=== OVERALL ===
Hard MIX (VAL)         R2= 0.811061  MAE= 0.033538  RMSE= 0.043779
Soft MIX (VAL)         R2= 0.812268  MAE= 0.033442  RMSE= 0.043639

Hard MIX (TEST)        R2= 0.723596  MAE= 0.038543  RMSE= 0.050505
Soft MIX (TEST)        R2= 0.724468  MAE= 0.038446  RMSE= 0.050425

=== WET ONLY ===
Hard MIX wet (VAL)     R2= 0.368471  MAE= 0.034377  RMSE= 0.044688
Soft MIX wet (VAL)     R2= 0.370624  MAE= 0.034302  RMSE= 0.044611

Hard MIX wet (TEST)    R2= 0.163596  MAE= 0.042397  RMSE= 0.053111
Soft MIX wet (TEST)    R2= 0.161911  MAE= 0.042409  RMSE= 0.053164

=== DRY ONLY ===
Hard MIX dry (VAL)     R2= 0.818629  MAE= 0.032849  RMSE= 0.043020
Soft MIX dry (VAL)     R2= 0.820266  MAE= 0.032737  RMSE= 0.042826

Hard MIX dry (TEST)    R2= 0.769670  MAE= 0.035354  RMSE= 0.048243
Soft MIX dry (TEST)    R2= 0.771591  MAE= 0.035167  RMSE= 0.048041


In v9.2, I augmented the wet-regime expert with rain impulse features capturing event timing and short-horizon precipitation history. 

While this improved the standalone wet expert’s performance, the gains were insufficient to improve the overall gated mixture. 

This suggests that wet-regime soil moisture dynamics are not fully captured by shallow precipitation features alone, and likely require higher-capacity or explicitly temporal modeling.

---

_Jakob Balkovec_